# Saheli — Unsloth Fine-Tuning on Google Colab (Gemma 4 E4B)

Run cells in order. Use **Runtime → Change runtime type → T4 GPU** (free) or A100 (Pro).

### Before you start

Have the project zip ready (`saheli.zip`, built locally — see repo README). Upload via one of two paths:

- **Path A — Direct upload (one-shot session):** the Step 2 cell opens a file picker.
- **Path B — Google Drive (survives restarts):** put `saheli.zip` in your Drive and mount.

## Step 1 — Install Unsloth
⚠️ Restart the runtime after this cell, then continue from Step 2.

In [ ]:
# Official Unsloth install — works on T4 / A100 / L4
# Reference: https://unsloth.ai/docs/models/gemma-4/train
!pip install -q unsloth
!pip install -q --upgrade --no-deps unsloth
!pip install -q trl transformers datasets accelerate peft bitsandbytes huggingface_hub sentencepiece

import torch, transformers, trl, unsloth
print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('trl          :', trl.__version__)
print('unsloth      :', unsloth.__version__)
print('CUDA         :', torch.cuda.is_available())

## Step 2 — Upload the project zip

Pick ONE of the two cells below. The other can be skipped.

### Option A — Upload via file picker (simple, session-only)

In [ ]:
from google.colab import files
import os

os.chdir('/content')
print('Pick saheli.zip from your computer...')
uploaded = files.upload()   # browse → saheli.zip
print('Uploaded:', list(uploaded.keys()))

### Option B — Mount Google Drive (saheli.zip should live somewhere in My Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to wherever you stored saheli.zip in Drive
import shutil
src = '/content/drive/MyDrive/saheli.zip'
shutil.copy(src, '/content/saheli.zip')
print('Copied to /content/saheli.zip')

## Step 3 — Unpack and enter the project

In [ ]:
import os, sys, zipfile, shutil

ZIP_PATH = '/content/saheli.zip'
WORKDIR  = '/content/saheli'

assert os.path.exists(ZIP_PATH), f'{ZIP_PATH} missing — re-run Step 2'

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR)

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(WORKDIR)

# Some zips wrap content in an extra top-level folder. Flatten if needed.
entries = os.listdir(WORKDIR)
if len(entries) == 1 and os.path.isdir(os.path.join(WORKDIR, entries[0])):
    inner = os.path.join(WORKDIR, entries[0])
    for e in os.listdir(inner):
        shutil.move(os.path.join(inner, e), os.path.join(WORKDIR, e))
    os.rmdir(inner)

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)

print('Working dir            :', os.getcwd())
print('finetune/train_unsloth :', os.path.exists('finetune/train_unsloth.py'))
print('data/who_anc_checklist :', os.path.exists('data/who_anc_checklist.json'))

## Step 4 — Configure training

In [ ]:
import os, torch

print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — STOP')
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → T4'

# ── Model ─────────────────────────────────────────────────────────────────────
os.environ['HF_MODEL_ID']   = 'unsloth/gemma-4-E4B-it'   # ← verify slug on HF

# ── Sequence length ───────────────────────────────────────────────────────────
# Hybrid output (JSON + reasoning) is longer than the old plain-text target.
# 1024 fits comfortably on T4 (16GB). Drop to 768 if you OOM.
os.environ['MAX_SEQ_LENGTH'] = '1024'

# ── Dataset location ──────────────────────────────────────────────────────────
os.environ['DATASET_PATH'] = './finetune/datasets/saheli_anc_dataset'

# ── Output paths (Colab writes to /content/ which is session-local) ───────────
os.environ['OUTPUT_DIR'] = '/content/outputs'
os.environ['MODEL_OUT']  = '/content/models/saheli-gemma4-e4b'

# ── Stability flags ───────────────────────────────────────────────────────────
os.environ['TORCH_COMPILE_DISABLE']      = '1'
os.environ['UNSLOTH_COMPILE_DISABLE']    = '1'
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'

print('HF_MODEL_ID   :', os.environ['HF_MODEL_ID'])
print('MAX_SEQ_LENGTH:', os.environ['MAX_SEQ_LENGTH'])
print('DATASET_PATH  :', os.environ['DATASET_PATH'])
print('MODEL_OUT     :', os.environ['MODEL_OUT'])

## Step 5 — Prepare dataset + run training

Two scripts, back-to-back:

1. `finetune/prepare_dataset.py` — generates 1,000 hybrid-format WHO ANC conversations (JSON tool call + plain-language RED/YELLOW/GREEN summary), with varied prompt templates, vitals, and paraphrases.
2. `finetune/train_unsloth.py` — loads the dataset, loads Gemma 4 E4B with `FastModel` (4-bit), applies LoRA (`r=16, alpha=32`) on language layers, trains 3 epochs with `train_on_responses_only`, exports GGUF Q4_K_M.

**Watch the `CHECK FORMATTED SAMPLE` output** near training start — it must contain `<start_of_turn>user` and `<start_of_turn>model`. Missing those = masking silently fails.

In [ ]:
import subprocess, sys

# Step 5a — generate dataset
prep_cmd = [sys.executable, 'finetune/prepare_dataset.py']
print('Running:', ' '.join(prep_cmd))
print('=' * 60)
ret = subprocess.call(prep_cmd, cwd='/content/saheli')
if ret != 0:
    raise RuntimeError('Dataset generation failed')

# Step 5b — run training
train_cmd = [sys.executable, 'finetune/train_unsloth.py']
print('Running:', ' '.join(train_cmd))
print('=' * 60)
proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd='/content/saheli',
)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()

print('=' * 60)
print('Exit code:', ret)
if ret != 0:
    raise RuntimeError('Training failed — check output above.')
print('Training complete!')

## Step 6 — Verify output files

In [ ]:
import os

MODEL_OUT = '/content/models/saheli-gemma4-e4b'
print('Output files:')
for root, _, files in os.walk(MODEL_OUT):
    for f in files:
        full = os.path.join(root, f)
        size = os.path.getsize(full) / 1e6
        print(f'  {os.path.relpath(full, MODEL_OUT):50s} {size:8.1f} MB')

## Step 7 — Package for download

Zips the adapter + GGUF so you can pull it off Colab before the runtime dies.

In [ ]:
import os, zipfile

MODEL_OUT = '/content/models/saheli-gemma4-e4b'
ZIP_PATH  = '/content/saheli-gemma4-e4b.zip'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(MODEL_OUT):
        for f in files:
            full = os.path.join(root, f)
            rel  = os.path.relpath(full, MODEL_OUT)
            zf.write(full, rel)

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Zip created: {ZIP_PATH}  ({size_mb:.1f} MB)')

### Option A — Download zip directly to your computer

In [ ]:
from google.colab import files
files.download('/content/saheli-gemma4-e4b.zip')

### Option B — Copy zip into your Google Drive (survives session end)

In [ ]:
import shutil
# Make sure Drive is mounted (Step 2, Option B). Then:
shutil.copy('/content/saheli-gemma4-e4b.zip', '/content/drive/MyDrive/saheli-gemma4-e4b.zip')
print('Saved to Drive: MyDrive/saheli-gemma4-e4b.zip')

## Step 8 — Push to HuggingFace (needed for Unsloth prize track)

Store your HF token with `userdata` (Colab's secret manager): left sidebar → 🔑 → add `HF_TOKEN`.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, create_repo
import os

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise RuntimeError('No HF_TOKEN — add it under 🔑 in the left sidebar, or skip this cell.')

HF_USERNAME = 'yourusername'        # ← set your HF username
HF_REPO     = 'saheli-gemma4-e4b'
MODEL_OUT   = '/content/models/saheli-gemma4-e4b'

api = HfApi(token=HF_TOKEN)
create_repo(f'{HF_USERNAME}/{HF_REPO}', token=HF_TOKEN, exist_ok=True)
api.upload_folder(
    folder_path = MODEL_OUT,
    repo_id     = f'{HF_USERNAME}/{HF_REPO}',
    repo_type   = 'model',
)
print(f'Pushed to https://huggingface.co/{HF_USERNAME}/{HF_REPO}')

## Step 9 — Evaluate (optional, needs full project)

In [ ]:
import os, subprocess, sys

eval_script = 'finetune/evaluate_model.py'
if os.path.exists(eval_script):
    ret = subprocess.call([sys.executable, eval_script])
    print('Evaluation exit code:', ret)
else:
    print('evaluate_model.py not found — re-run Step 3 to unzip the project.')